# 58 — Distillation Project (Teacher-Student)

## Goal
Use a **stronger model** (teacher) to generate high-quality training data,
then fine-tune a **smaller model** (student) on that data.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Knowledge distillation** | Transfer knowledge from large → small model |
| **Synthetic data generation** | Use teacher to create training data |
| **Quality filtering** | Filter teacher outputs before training |
| **Model comparison** | Benchmark student vs. teacher quality |

## Pipeline
```
Task Prompts → Teacher (large) → Synthetic Data → Filter → Student (small) SFT
```

## Stack
- **Teacher**: Ollama `qwen3:4b` (larger model)
- **Student**: `Qwen2.5-0.5B-Instruct` (smaller, fine-tunable)
- `transformers` + `peft` + `trl`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import torch
import ollama
print(f"PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}")

## Step 1 — Define Task & Generate Prompts

Task: **Code explanation** — explain a code snippet in plain English.
We create diverse prompts, then use the teacher to generate explanations.

In [ ]:
code_snippets = [
    {"code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)", "language": "Python"},
    {"code": "nums = [1, 2, 3, 4, 5]\nresult = list(filter(lambda x: x % 2 == 0, nums))", "language": "Python"},
    {"code": "from collections import Counter\nwords = text.split()\nword_freq = Counter(words).most_common(10)", "language": "Python"},
    {"code": "def binary_search(arr, target):\n    lo, hi = 0, len(arr) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == target: return mid\n        elif arr[mid] < target: lo = mid + 1\n        else: hi = mid - 1\n    return -1", "language": "Python"},
    {"code": "with open('data.csv') as f:\n    reader = csv.DictReader(f)\n    for row in reader:\n        print(row['name'], row['age'])", "language": "Python"},
    {"code": "class Stack:\n    def __init__(self):\n        self.items = []\n    def push(self, item):\n        self.items.append(item)\n    def pop(self):\n        return self.items.pop()", "language": "Python"},
    {"code": "import re\npattern = r'\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Z|a-z]{2,}\\b'\nemails = re.findall(pattern, text)", "language": "Python"},
    {"code": "squares = {x: x**2 for x in range(10)}\nprint(squares[7])  # output: 49", "language": "Python"},
    {"code": "from functools import lru_cache\n\n@lru_cache(maxsize=128)\ndef expensive_computation(n):\n    return sum(i**2 for i in range(n))", "language": "Python"},
    {"code": "try:\n    result = int(user_input)\nexcept ValueError:\n    print('Please enter a valid number')\nfinally:\n    print('Input processing complete')", "language": "Python"},
    {"code": "from contextlib import contextmanager\n\n@contextmanager\ndef timer():\n    import time\n    start = time.time()\n    yield\n    print(f'Elapsed: {time.time()-start:.2f}s')", "language": "Python"},
    {"code": "data = sorted(students, key=lambda s: s['gpa'], reverse=True)\ntop_3 = data[:3]", "language": "Python"},
    {"code": "async def fetch_data(url):\n    async with aiohttp.ClientSession() as session:\n        async with session.get(url) as response:\n            return await response.json()", "language": "Python"},
    {"code": "matrix = [[1,2,3],[4,5,6],[7,8,9]]\ntransposed = list(zip(*matrix))", "language": "Python"},
    {"code": "from dataclasses import dataclass\n\n@dataclass\nclass Point:\n    x: float\n    y: float\n    \n    def distance(self, other):\n        return ((self.x - other.x)**2 + (self.y - other.y)**2)**0.5", "language": "Python"},
]

print(f"Created {len(code_snippets)} code snippets for teacher labeling")

## Step 2 — Teacher Generates Training Data

We ask the teacher (qwen3:4b) to explain each code snippet.
This creates synthetic training data for the student.

In [ ]:
TEACHER_PROMPT = """Explain this {language} code in plain English for a beginner.
Be concise (3-5 sentences). Cover:
1. What the code does (purpose)
2. How it works (key mechanism)
3. When you'd use it (practical context)

Code:
```{language}
{code}
```

Explanation:"""

teacher_outputs = []
print("Generating teacher explanations...\n")

for i, snippet in enumerate(code_snippets):
    prompt = TEACHER_PROMPT.format(**snippet)
    resp = ollama.chat(
        model="qwen3:4b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.3, "num_predict": 300},
    )
    explanation = resp["message"]["content"].strip()

    teacher_outputs.append({
        "code": snippet["code"],
        "language": snippet["language"],
        "explanation": explanation,
    })
    print(f"  [{i+1}/{len(code_snippets)}] {snippet['code'][:40]}... → {len(explanation)} chars")

print(f"\n✅ Teacher generated {len(teacher_outputs)} explanations")

## Step 3 — Quality Filtering

Not all teacher outputs are good. Filter out:
- Too short (< 50 chars) — probably incomplete
- Too long (> 1000 chars) — probably rambling
- Empty or error responses

In [ ]:
def quality_filter(example):
    explanation = example["explanation"]
    if len(explanation) < 50:
        return False, "too short"
    if len(explanation) > 1500:
        return False, "too long"
    if "error" in explanation.lower() and "traceback" in explanation.lower():
        return False, "contains error"
    return True, "ok"

filtered = []
rejected = []
for ex in teacher_outputs:
    passed, reason = quality_filter(ex)
    if passed:
        filtered.append(ex)
    else:
        rejected.append((ex, reason))

print(f"Passed: {len(filtered)}/{len(teacher_outputs)}")
print(f"Rejected: {len(rejected)}")
for ex, reason in rejected:
    print(f"  ❌ {reason}: {ex['code'][:40]}...")

In [ ]:
# Inspect a few teacher outputs
for i, ex in enumerate(filtered[:3]):
    print(f"\n{'='*60}")
    print(f"Code: {ex['code'][:60]}...")
    print(f"Teacher: {ex['explanation'][:200]}...")

## Step 4 — Fine-Tune Student on Teacher Data

In [ ]:
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

STUDENT_ID = "Qwen/Qwen2.5-0.5B-Instruct"
OUTPUT_DIR = "./distilled_code_explainer"

SYS_MSG = "You are a code explainer. Given a code snippet, explain what it does in plain English. Be concise (3-5 sentences)."

student_data = []
for ex in filtered:
    student_data.append({"messages": [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Explain this code:\n```{ex['language']}\n{ex['code']}\n```"},
        {"role": "assistant", "content": ex["explanation"]},
    ]})

train_ds = Dataset.from_list(student_data)

trainer = SFTTrainer(
    model=STUDENT_ID,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        max_steps=60,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=5,
        max_length=512,
        bf16=True,
        logging_steps=10,
        gradient_checkpointing=True,
        report_to="none",
    ),
    train_dataset=train_ds,
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        task_type="CAUSAL_LM",
    ),
)

print(f"Distilling {len(filtered)} teacher examples into student...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print("✅ Distillation complete!")

## Step 5 — Compare Teacher vs Student

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

student_model = AutoPeftModelForCausalLM.from_pretrained(OUTPUT_DIR, torch_dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)

# Unseen test code
test_snippets = [
    """def flatten(lst):\n    result = []\n    for item in lst:\n        if isinstance(item, list):\n            result.extend(flatten(item))\n        else:\n            result.append(item)\n    return result""",
    """import hashlib\nhash_obj = hashlib.sha256(password.encode())\nhashed = hash_obj.hexdigest()""",
]

for code in test_snippets:
    print(f"\n{'='*70}")
    print(f"CODE: {code[:60]}...")
    print(f"{'='*70}")

    # Teacher
    prompt = TEACHER_PROMPT.format(language="Python", code=code)
    teacher_resp = ollama.chat(model="qwen3:4b", messages=[{"role": "user", "content": prompt}],
                               options={"temperature": 0.3, "num_predict": 300})
    print(f"\n🔵 TEACHER (qwen3:4b):")
    print(teacher_resp["message"]["content"][:300])

    # Student
    messages = [
        {"role": "system", "content": SYS_MSG},
        {"role": "user", "content": f"Explain this code:\n```python\n{code}\n```"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(student_model.device)
    with torch.no_grad():
        out = student_model.generate(**inputs, max_new_tokens=200, temperature=0.3, do_sample=True)
    student_resp = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    print(f"\n🟢 STUDENT (0.5B distilled):")
    print(student_resp[:300])

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Teacher** | Larger model (qwen3:4b) generates high-quality data |
| **Student** | Smaller model (0.5B) learns from teacher outputs |
| **Quality filter** | Remove too-short, too-long, or error outputs |
| **Distillation** | Student approximates teacher quality at lower cost |

## Distillation Pipeline
```
Task Prompts → Teacher Model → Raw Data → Quality Filter → Clean Data → Student SFT
                   (4B)                                                     (0.5B)
```

## Why Distill?
- **Cost**: Student is 8x smaller → 8x cheaper to serve
- **Latency**: Smaller model → faster inference
- **Privacy**: Student can run locally, teacher data stays offline
- **Quality**: Student gets ~80-90% of teacher quality for specific tasks